# 02 - Cleaning hasil Extract
Tujuan  : Structure-preserving cleaning
Prinsip : Hanya normalisasi dan hapus artefak ekstraksi. Semua penanda struktur (Markdown, layout_classes) dipertahankan untuk Notebook 03 (structuring). Keputusan drop/keep halaman dilakukan di 04A/04B.

In [1]:
import re
import json
import unicodedata

from pathlib import Path
from typing import Dict, List, Tuple

## 1. Konfigurasi

In [2]:
EXTRACT_DIR = Path("/kaggle/input/notebooks/muhammadnabilul/01-extract-pdf-pymupdfllm-ocr/extracted")
CLEAN_DIR   = Path("/kaggle/working/cleaned")
CLEAN_DIR.mkdir(parents=True, exist_ok=True,)

# Versi pipeline — naikkan saat ada perubahan signifikan
PIPELINE_VERSION: str = "v1"

# Jika True, teks di dalam OCR picture block dipertahankan
# (hanya marker Start/End yang dihapus, bukan isinya).
# Jika False, seluruh block dihapus.
# Default: False — isi OCR gambar umumnya rusak strukturnya.
KEEP_PICTURE_TEXT: bool = True

In [3]:
def load_json(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        return json.load(f)
 
 
def save_json(data: dict, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

### Preview

In [4]:
json_files = []

for file in sorted(EXTRACT_DIR.glob("*.json")):
    try:
        data = load_json(file)
        if (
            isinstance(data, dict)
            and "pages" in data
        ):
            json_files.append(file)
    except Exception:
        pass
        
print(f"Dokumen valid: {len(json_files)}")

for i, file in enumerate(json_files): 
    print(f"[{i+1:>3}] {file.name}")

Dokumen valid: 47
[  1] analisis-petty-corruption-hasil-spak-2020---2024.json
[  2] indeks-pembangunan-manusia-2022.json
[  3] indeks-pembangunan-manusia-2023.json
[  4] indeks-pembangunan-manusia-2024.json
[  5] indeks-perilaku-anti-korupsi-2023.json
[  6] indeks-perilaku-anti-korupsi-2024.json
[  7] indikator-kesejahteraan-rakyat-2023.json
[  8] indikator-kesejahteraan-rakyat-2024.json
[  9] indikator-kesejahteraan-rakyat-2025.json
[ 10] indikator-sdgs-kesejahteraan-rakyat-2024.json
[ 11] indikator-sdgs-kesejahteraan-rakyat-2025.json
[ 12] infografis-statistik-kesejahteraan-rakyat-2025.json
[ 13] kesejahteraan-anak-indonesia--analisis-deprivasi-hak-anak-multidimensi-2022.json
[ 14] kesejahteraan-anak-indonesia--analisis-kemiskinan-anak-moneter-2022.json
[ 15] laporan-indeks-khusus-penanganan-stunting--ikps--2022-2023.json
[ 16] laporan-indeks-khusus-penanganan-stunting-2021-2022.json
[ 17] laporan-indeks-khusus-penanganan-stunting-kabupaten-kota-2021-2022.json
[ 18] profil-anak-usia-

In [5]:
sample_doc = load_json(json_files[0])
print(f"Dokumen : {json_files[0].name}")
print(f"Halaman : {len(sample_doc['pages'])}")
print("\nPreview text_md:\n")
print(sample_doc["pages"][0]["text_md"][:1000])

Dokumen : analisis-petty-corruption-hasil-spak-2020---2024.json
Halaman : 194

Preview text_md:

# WASIL SPAK D000 oat 




## 2. Konstanta pembersihan

In [6]:
# Watermark BPS — semua variant yang ditemukan di korpus
WATERMARK_PATTERNS = [
    r"https?://www\.bps\.go\.id",
    r"//www\.bps\.go\.id",
    r"/www\.bps\.go\.id",
    r"www\.bps\.go\.id",
    r"https?://bps\.go\.id",
    r"bpshq@bps\.go\.id",
]

# Marker picture block dari pymupdf4llm
# (dengan dan tanpa bold markdown)
PICTURE_BLOCK_START = re.compile(
    r"\*{0,2}-{5}\s*Start of picture text\s*-{5}\*{0,2}",
    re.IGNORECASE,
)
PICTURE_BLOCK_END = re.compile(
    r"\*{0,2}-{5}\s*End of picture text\s*-{5}\*{0,2}",
    re.IGNORECASE,
)

# Picture omitted marker
PICTURE_OMITTED_PATTERN = re.compile(
    r"\*{0,2}==>\s*picture.*?<==\*{0,2}",
    re.IGNORECASE,
)

# HTML artifact dari pymupdf4llm
BR_PATTERN = re.compile(r"<br\s*/?>", re.IGNORECASE)

## 3. Fungsi-Fungsi

### a. Text Normalisasi

In [7]:
def normalize_unicode(text: str) -> str:
    """
    Normalisasi karakter unicode menggunakan NFKC.

    Menangani:
    - Ligature: fi, fl, ffi → karakter terpisah
    - Spasi non-breaking → spasi biasa
    - Karakter komposit → bentuk kanonik
    """
    return unicodedata.normalize("NFKC", text)

In [8]:
def normalize_whitespace(text: str) -> str:
    """
    Normalisasi spasi dan newline berlebih.

    Menangani:
    - Spasi/tab ganda → satu spasi
    - Lebih dari dua newline berurutan → dua newline
    """
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [9]:
def fix_hyphenation(text: str) -> str:
    """
    Memperbaiki kata yang terpotong akibat line-wrapping PDF.

    Hanya menangani huruf (bukan angka) untuk menghindari
    penggabungan rentang tahun seperti 2023-\\n2024 yang
    seharusnya tetap menjadi 2023–2024, bukan 20232024.

    Contoh yang ditangani:
        pendi-          →   pendidikan
        dikan

        statis-         →   statistik
        tik
    """
    return re.sub(
        r"([A-Za-zÀ-ÿ]+)-\n([A-Za-zÀ-ÿ]+)",
        r"\1\2",
        text,
    )

### b. Removal

In [10]:
def remove_br_tags(text: str) -> str:
    """
    Menghapus HTML <br> tag yang menjadi artifact
    dari proses ekstraksi pymupdf4llm.

    Contoh:
        teks pertama<br>teks kedua   →   teks pertama teks kedua
    """
    return BR_PATTERN.sub(" ", text)

In [11]:
def remove_picture_markers(
    text: str,
    keep_picture_text: bool = KEEP_PICTURE_TEXT,
) -> Tuple[str, int]:
    """
    Menghapus artefak gambar dari proses ekstraksi pymupdf4llm.

    Args:
        text              : teks input
        keep_picture_text : jika True, hanya hapus marker Start/End
                            dan pertahankan isi OCR di dalamnya.
                            Jika False (default), hapus seluruh block.

    Returns:
        (text_bersih, jumlah_block_dihapus)

    Catatan:
        Watermark BPS hampir selalu berada di dalam picture text
        block. Dengan menghapus seluruh block, watermark ikut
        terhapus tanpa regex tambahan.
    """
    blocks_removed = 0

    if keep_picture_text:
        text = PICTURE_BLOCK_START.sub("", text)
        text = PICTURE_BLOCK_END.sub("", text)
    else:
        while True:
            m_start = PICTURE_BLOCK_START.search(text)
            if not m_start:
                break
            m_end = PICTURE_BLOCK_END.search(text, m_start.end())
            if not m_end:
                text = text[:m_start.start()]
                blocks_removed += 1
                break
            text = text[:m_start.start()] + text[m_end.end():]
            blocks_removed += 1

    omitted_count = len(PICTURE_OMITTED_PATTERN.findall(text))
    text = PICTURE_OMITTED_PATTERN.sub("", text)
    blocks_removed += omitted_count

    return text, blocks_removed

In [12]:
def remove_bps_watermark(text: str) -> Tuple[str, int]:
    """
    Menghapus sisa watermark URL BPS yang tidak berada
    di dalam picture text block (standalone watermark).

    Dipanggil setelah remove_picture_markers() untuk
    menangkap variant yang lolos.

    Returns:
        (text_bersih, jumlah_kemunculan_dihapus)
    """
    total_removed = 0
    for pattern in WATERMARK_PATTERNS:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        total_removed += len(matches)
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    return text, total_removed

### c. Markdown Feature 

In [13]:
def extract_markdown_features(text: str) -> dict:
    """
    Ekstraksi fitur Markdown secara ringan.

    Catatan:
        Hanya mencatat keberadaan fitur — tidak melakukan
        interpretasi atau klasifikasi struktur dokumen.
        Interpretasi dilakukan di Notebook 03.
    """
    heading_levels = []
    for line in text.splitlines():
        match = re.match(r"^(#{1,6})\s+", line.strip())
        if match:
            heading_levels.append(len(match.group(1)))

    table_rows = len(
        re.findall(r"^\|.*\|$", text, re.MULTILINE)
    )

    return {
        "heading_count"     : len(heading_levels),
        "heading_levels"    : heading_levels,
        "table_row_count"   : table_rows,
        "has_markdown_table": table_rows > 0,
    }

### d. Cleaning metrix

In [14]:
def assess_clean_text(text: str) -> dict:
    """
    Metrik dasar hasil cleaning.

    Catatan:
        is_low_char hanya sebagai flag observasi untuk
        membantu analisis di Notebook 04A/04B.
        Keputusan drop/keep halaman tidak dilakukan di sini.
    """
    lines     = text.splitlines()
    non_empty = [l for l in lines if l.strip()]
    n_chars   = len(text.strip())

    return {
        "char_count"          : n_chars,
        "line_count"          : len(lines),
        "non_empty_line_count": len(non_empty),
        "is_empty"            : n_chars == 0,
        "is_low_char"         : 0 < n_chars < 100,
    }

## 4. Pipeline Cleaning

In [15]:
def clean_text(
    text: str,
    keep_picture_text: bool = KEEP_PICTURE_TEXT,
) -> Tuple[str, dict]:
    """
    Pipeline cleaning utama untuk satu string teks.

    Urutan operasi:
        1. normalize_unicode       — fix encoding & ligature
        2. fix_hyphenation         — sambung kata terpotong
        3. remove_br_tags          — hapus HTML <br> artifact
        4. remove_picture_markers  — hapus OCR block & placeholder gambar
        5. remove_bps_watermark    — hapus sisa watermark standalone
        6. normalize_whitespace    — normalisasi spasi & newline

    Returns:
        (text_bersih, cleaning_stats)
    """
    text = normalize_unicode(text)
    text = fix_hyphenation(text)
    text = remove_br_tags(text)

    text, picture_blocks_removed = remove_picture_markers(
        text, keep_picture_text=keep_picture_text,
    )
    text, watermark_removed = remove_bps_watermark(text)

    text = normalize_whitespace(text)

    cleaning_stats = {
        "picture_blocks_removed": picture_blocks_removed,
        "watermark_removed"     : watermark_removed,
    }

    return text, cleaning_stats

In [16]:
def clean_page(
    page: Dict,
    keep_picture_text: bool = KEEP_PICTURE_TEXT,
) -> Dict:
    """
    Cleaning satu halaman dokumen.

    Input  : page dict dari hasil ekstraksi Notebook 01
    Output : page dict yang diperkaya dengan:
             - text_raw          : teks asli sebelum cleaning
             - text_clean        : teks setelah cleaning
             - markdown_features : fitur Markdown
             - cleaning_metrics  : metrik hasil cleaning
             - cleaning_stats    : jumlah artefak yang dihapus

    Catatan:
        Semua field original (termasuk layout_classes) dipertahankan
        untuk digunakan di Notebook 03 (structuring).
    """
    text_raw = page["text_md"]
    text_clean, cleaning_stats = clean_text(
        text_raw, keep_picture_text=keep_picture_text,
    )

    return {
        **page,
        "text_raw"          : text_raw,
        "text_clean"        : text_clean,
        "markdown_features" : extract_markdown_features(text_clean),
        "cleaning_metrics"  : assess_clean_text(text_clean),
        "cleaning_stats"    : cleaning_stats,
    }


In [17]:
def clean_document(
    pages: List[Dict],
    keep_picture_text: bool = KEEP_PICTURE_TEXT,
) -> List[Dict]:
    """Cleaning seluruh halaman satu dokumen."""
    return [
        clean_page(page, keep_picture_text=keep_picture_text)
        for page in pages
    ]

## 5. Cleaning Summary

In [18]:
def build_cleaning_summary(cleaned_pages: List[Dict]) -> dict:
    """
    Ringkasan statistik hasil cleaning satu dokumen.
    Digunakan untuk quality report di akhir Notebook 02.

    Metrik reduction_pct dapat langsung dikutip di Bab 4:
        "Cleaning mengurangi ukuran korpus sebesar X%
         tanpa menghilangkan struktur dokumen."
    """
    chars = [
        p["cleaning_metrics"]["char_count"]
        for p in cleaned_pages
    ]

    total_chars_raw   = sum(len(p.get("text_raw", "")) for p in cleaned_pages)
    total_chars_clean = sum(chars)
    reduction_pct     = (
        (1 - total_chars_clean / total_chars_raw) * 100
        if total_chars_raw > 0 else 0.0
    )

    return {
        "total_pages"            : len(cleaned_pages),
        "empty_pages"            : sum(p["cleaning_metrics"]["is_empty"]    for p in cleaned_pages),
        "low_char_pages"         : sum(p["cleaning_metrics"]["is_low_char"] for p in cleaned_pages),
        "heading_pages"          : sum(p["markdown_features"]["heading_count"] > 0 for p in cleaned_pages),
        "table_pages"            : sum(p["markdown_features"]["has_markdown_table"] for p in cleaned_pages),
        "total_chars_raw"        : total_chars_raw,
        "total_chars_clean"      : total_chars_clean,
        "max_chars_page"         : max(chars) if chars else 0,
        "min_chars_page"         : min(chars) if chars else 0,
        "reduction_pct"          : round(reduction_pct, 2),
        "picture_blocks_removed" : sum(p["cleaning_stats"]["picture_blocks_removed"] for p in cleaned_pages),
        "watermark_removed"      : sum(p["cleaning_stats"]["watermark_removed"]       for p in cleaned_pages),
    }

## 6. Main Program

In [19]:
def process_file(
    input_path: Path,
    output_dir: Path,
    keep_picture_text: bool = KEEP_PICTURE_TEXT,
    verbose: bool = True,
) -> Path:
    """
    Baca satu file JSON hasil ekstraksi (Notebook 01),
    jalankan cleaning pipeline, simpan hasil ke output_dir.

    Struktur output JSON sama dengan input, diperkaya dengan:
        - text_raw, text_clean, markdown_features,
          cleaning_metrics, cleaning_stats (per halaman)
        - cleaning_config, cleaning_summary (level dokumen)

    Returns:
        Path file output yang dihasilkan
    """
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    doc   = load_json(input_path)
    pages = doc.get("pages", [])
    meta  = {k: v for k, v in doc.items() if k != "pages"}

    cleaned_pages = clean_document(pages, keep_picture_text=keep_picture_text)
    summary       = build_cleaning_summary(cleaned_pages)

    output = {
        **meta,
        "cleaning_config": {
            "pipeline_version" : PIPELINE_VERSION,
            "keep_picture_text": keep_picture_text,
            "notebook"         : "02_cleaning",
        },
        "cleaning_summary": summary,
        "pages"           : cleaned_pages,
    }

    out_path = output_dir / input_path.name
    save_json(output, out_path)

    if verbose:
        print(f"[OK] {input_path.name}")
        print(f"     halaman          : {summary['total_pages']}")
        print(f"     chars raw        : {summary['total_chars_raw']:,}")
        print(f"     chars clean      : {summary['total_chars_clean']:,}")
        print(f"     reduction        : {summary['reduction_pct']}%")
        print(f"     max chars/page   : {summary['max_chars_page']:,}")
        print(f"     min chars/page   : {summary['min_chars_page']:,}")
        print(f"     picture removed  : {summary['picture_blocks_removed']}")
        print(f"     watermark removed: {summary['watermark_removed']}")
        print(f"     empty pages      : {summary['empty_pages']}")
        print(f"     low_char pages   : {summary['low_char_pages']}")
        print(f"     output           : {out_path}")

    return out_path

In [20]:
def process_directory(
    json_files: List[Path],
    output_dir: Path,
    keep_picture_text: bool = KEEP_PICTURE_TEXT,
    verbose: bool = True,
) -> List[Path]:
    """
    Memproses daftar file JSON yang sudah lolos validasi.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not json_files:
        print("[WARN] Tidak ada dokumen valid.")
        return []

    if verbose:
        print(f"Ditemukan {len(json_files)} dokumen valid")
        print(f"Output ke : {output_dir}\n")

    results = []

    for i, json_file in enumerate(json_files, start=1):

        if verbose:
            print(
                f"[{i}/{len(json_files)}] "
                f"{json_file.name}"
            )

        try:
            out = process_file(
                json_file,
                output_dir,
                keep_picture_text=keep_picture_text,
                verbose=verbose,
            )

            results.append(out)

        except Exception as e:
            print(
                f"[ERROR] {json_file.name}"
            )
            print(f"        {e}")

        if verbose:
            print()

    if verbose:
        print(
            f"Selesai. "
            f"{len(results)} file disimpan."
        )

    return results

In [21]:
results = process_directory(
    json_files=json_files,
    output_dir=CLEAN_DIR,
    keep_picture_text=KEEP_PICTURE_TEXT,
    verbose=True,
)

print(
    f"\nSelesai membersihkan "
    f"{len(results):,} dokumen."
)

Ditemukan 47 dokumen valid
Output ke : /kaggle/working/cleaned

[1/47] analisis-petty-corruption-hasil-spak-2020---2024.json
[OK] analisis-petty-corruption-hasil-spak-2020---2024.json
     halaman          : 194
     chars raw        : 474,091
     chars clean      : 450,612
     reduction        : 4.95%
     max chars/page   : 11,070
     min chars/page   : 0
     picture removed  : 175
     watermark removed: 54
     empty pages      : 17
     low_char pages   : 5
     output           : /kaggle/working/cleaned/analisis-petty-corruption-hasil-spak-2020---2024.json

[2/47] indeks-pembangunan-manusia-2022.json
[OK] indeks-pembangunan-manusia-2022.json
     halaman          : 180
     chars raw        : 368,491
     chars clean      : 317,355
     reduction        : 13.88%
     max chars/page   : 7,885
     min chars/page   : 0
     picture removed  : 240
     watermark removed: 146
     empty pages      : 17
     low_char pages   : 5
     output           : /kaggle/working/cleaned/inde